# AeroTwin AI – Notebook 01

**Objetivo:** Construir un gemelo digital (digital twin) del ecosistema aeronáutico español utilizando datos abiertos de AENA y ENAIRE junto con técnicas de IA generativa (RAG).

**Fuentes de datos:**
- AENA – estadísticas de pasajeros (`data/raw/aena_passengers/`)
- AENA – directorio de aerolíneas (`data/raw/aena_airlines/`)
- AENA – metadatos de aeropuertos (`data/raw/aena_airports/`)
- ENAIRE – publicación de información aeronáutica AIP (`data/raw/enaire_aip/`)

## 0. Configuración del entorno

In [21]:
!pip install -r requirements.txt

In [24]:
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Pega aquí tu Google Gemini API Key: ")

print("API key configurada correctamente:", bool(os.environ.get("GOOGLE_API_KEY")))

API key configurada correctamente: True


In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

response = llm.invoke("Responde solo con esta frase: Gemini funciona correctamente.")

print(response.content)

Gemini funciona correctamente.


In [27]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    output_dimensionality=768
)

test_vector = embeddings.embed_query("AeroTwin AI test embedding")

print("Embedding generado correctamente.")
print("Dimensión del vector:", len(test_vector))
print("Primeros 5 valores:", test_vector[:5])

Embedding generado correctamente.
Dimensión del vector: 768
Primeros 5 valores: [-0.037928145, 0.00288188, -0.017322036, -0.011273562, -0.06233103]


In [28]:
%cd /content/aerotwin-ai

!git pull

print("\nContenido actual de .env.example:")
with open(".env.example", "r", encoding="utf-8") as f:
    print(f.read())

/content/aerotwin-ai
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.08 KiB | 1.08 MiB/s, done.
From https://github.com/mg15best/aerotwin-ai
   d09c0cd..c1afcaa  main       -> origin/main
Updating d09c0cd..c1afcaa
Fast-forward
 ...ide_july_2026.pdf.pdf => aena_price_guide_july_2026.pdf} | Bin
 1 file changed, 0 insertions(+), 0 deletions(-)
 rename data/raw/aena_airlines/{aena_price_guide_july_2026.pdf.pdf => aena_price_guide_july_2026.pdf} (100%)

Contenido actual de .env.example:
# Google Gemini API Key
# Required to run AeroTwin AI with Gemini LLM and Gemini Embeddings
GOOGLE_API_KEY=your_google_gemini_api_key_here

# Optional model configuration
GEMINI_MODEL=gemini-2.5-flash
GEMINI_EMBEDDING_MODEL=gemini-embedding-2-preview
GEMINI_EMBEDDING_DIMENSIONS=768



In [29]:
from pathlib import Path

RAW_DATA_DIR = Path("data/raw")

document_files = []

for path in RAW_DATA_DIR.rglob("*"):
    if path.is_file():
        if "support_optional" in str(path):
            continue
        if path.name == ".gitkeep":
            continue
        if path.suffix.lower() in [".pdf", ".html", ".htm"]:
            document_files.append(path)

print("Documentos principales encontrados para el RAG:", len(document_files))

for file in document_files:
    print("-", file)

Documentos principales encontrados para el RAG: 18
- data/raw/aena_passengers/aena_tfs_information_meeting_points.html
- data/raw/aena_passengers/aena_passengers_general.html
- data/raw/aena_passengers/aena_tfs_special_needs.html
- data/raw/aena_passengers/aena_mad_passenger_homepage.html
- data/raw/aena_passengers/aena_passengers_prm_general.html
- data/raw/aena_passengers/aena_mad_airport_guide_maps.html
- data/raw/aena_airlines/aena_price_guide_july_2026.pdf
- data/raw/aena_airlines/aena_mad_business_profile.html
- data/raw/aena_airlines/aena_airlines_marketing_support.html
- data/raw/aena_airlines/aena_airlines_portal.html
- data/raw/aena_airlines/aena_airlines_new_routes.html
- data/raw/aena_airlines/aena_airlines_commercial_incentives.html
- data/raw/aena_airlines/aena_airports_destinations_network.html
- data/raw/enaire_aip/enaire_aip_gcxo_tenerife_norte.pdf
- data/raw/enaire_aip/enaire_aip_spain_general.html
- data/raw/enaire_aip/enaire_aip_lemd_madrid_barajas.html
- data/raw/e

## 1. Carga de datos

In [ ]:
from src.loaders import (
    load_aena_passengers,
    load_aena_airlines,
    load_aena_airports,
    load_enaire_aip_texts,
)

# Descomentar cuando los datos estén disponibles en data/raw/
# df_passengers = load_aena_passengers()
# df_airlines   = load_aena_airlines()
# df_airports   = load_aena_airports()
# aip_docs      = load_enaire_aip_texts()

print("Loaders importados correctamente.")

## 2. Análisis exploratorio de datos (EDA)

> Ejecutar esta sección una vez que los datos estén disponibles en `data/raw/`.

In [ ]:
# Ejemplo de análisis exploratorio (completar con datos reales)
# print(df_passengers.shape)
# df_passengers.head()
print("EDA – pendiente de datos.")

## 3. Indexación de documentos AIP (Vector Store)

Construimos un índice FAISS con embeddings de OpenAI sobre los documentos AIP de ENAIRE.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

# Ejemplo: crear documentos LangChain desde los textos AIP
# raw_docs = load_enaire_aip_texts()
# lc_docs = [Document(page_content=d["content"], metadata={"source": d["source"]}) for d in raw_docs]

# splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# chunks = splitter.split_documents(lc_docs)

# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# vector_store = FAISS.from_documents(chunks, embeddings)
# vector_store.save_local(ROOT / "data" / "processed" / "faiss_aip_index")

print("Indexación – pendiente de documentos AIP en data/raw/enaire_aip/.")

## 4. Pipeline RAG – AeroTwin AI

Combinamos el contexto extraído del vector store con un LLM para responder preguntas sobre el ecosistema aeronáutico.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

SYSTEM_PROMPT = """
Eres AeroTwin AI, un asistente especializado en el ecosistema aeronáutico español.
Utiliza únicamente el contexto proporcionado para responder las preguntas.
Si no conoces la respuesta, di que no tienes información suficiente.

Contexto:
{context}

Pregunta: {question}
Respuesta:"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=SYSTEM_PROMPT,
)

# llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0)

# Cargar índice guardado
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# vector_store = FAISS.load_local(
#     ROOT / "data" / "processed" / "faiss_aip_index",
#     embeddings,
#     allow_dangerous_deserialization=True,
# )

# qa_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="stuff",
#     retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
#     chain_type_kwargs={"prompt": prompt},
# )

print("Pipeline RAG – pendiente de índice vectorial.")

## 5. Preguntas de demostración

Ejecuta las preguntas del fichero `outputs/demo_questions.md` contra el pipeline RAG.

In [ ]:
demo_questions = [
    "¿Cuál fue el aeropuerto español con mayor tráfico de pasajeros en 2023?",
    "¿Cuáles son los procedimientos de llegada instrumental (STAR) vigentes en Madrid-Barajas?",
    "¿Qué aerolíneas operan más rutas en los aeropuertos de AENA?",
]

for question in demo_questions:
    print(f"\n❓ {question}")
    # answer = qa_chain.run(question)
    # print(f"💡 {answer}")
    print("   → (pipeline RAG pendiente de datos)")

## 6. Próximos pasos

1. Descargar los datasets de AENA desde el [portal de datos abiertos](https://estadisticas.aena.es/estadisticas/SASPortal/main.do) y colocarlos en `data/raw/aena_passengers/`, `data/raw/aena_airlines/` y `data/raw/aena_airports/`.
2. Descargar los documentos AIP de ENAIRE desde [su portal oficial](https://aip.enaire.es/) y colocarlos en `data/raw/enaire_aip/`.
3. Completar la celda de indexación (sección 3) y ejecutarla para construir el índice FAISS.
4. Probar el pipeline RAG con las preguntas de demostración (sección 5).
5. Evaluar la calidad de las respuestas y ajustar los parámetros de recuperación.